# CS766 — Midterm Analysis
**Evaluating Semantic Grounding in Open-Vocabulary Segmentation**

Authors: Yutesh Vishnu Addanki · Dinesh Kumar Rajakumar  
Date: March 25, 2025  
Course: CS766 Computer Vision — University of Wisconsin-Madison

---

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # run from notebooks/ folder

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image

plt.rcParams.update({
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
})

RESULTS_DIR    = Path('../results')
COMPARISON_DIR = RESULTS_DIR / 'comparison'
MC_DIR         = RESULTS_DIR / 'maskclip'
DC_DIR         = RESULTS_DIR / 'denseclip'
EVAL_JSON      = RESULTS_DIR / 'full_eval.json'

print('Setup complete.')

---
## 1. Architecture Comparison

| | MaskCLIP | DenseCLIP |
|---|---|---|
| Backbone | ViT-B/16 (frozen) | ViT-B/16 (frozen) |
| Dense head | None | ContextDecoder + Conv |
| Trainable params | **0** | ~6M |
| Segmentation signal | Cosine similarity of patch tokens | Cross-attention enriched features |
| Output | Soft similarity map | Sigmoid probability map |
| Speed | Faster (no dense head) | Slower (extra decoder) |

**Key difference:** MaskCLIP repurposes frozen CLIP patch tokens directly as a segmentation signal. DenseCLIP adds a trainable cross-attention decoder that injects language context into the visual features before prediction — enabling stronger spatial grounding at the cost of additional parameters.

---
## 2. Quantitative Results

In [ ]:
# Load evaluation results
# Run: python evaluation/evaluate.py --model all
# before executing this cell

if not EVAL_JSON.exists():
    print(f"[!] {EVAL_JSON} not found.")
    print("    Run: python evaluation/evaluate.py --model all")
else:
    with open(EVAL_JSON) as f:
        results = json.load(f)

    rows = []
    for model_name, res in results.items():
        agg = res['aggregate']
        rows.append({
            'Model':               model_name,
            'mIoU':                agg.get('miou',                    'N/A'),
            'CLIP Similarity':     agg.get('mean_clip_sim',           'N/A'),
            'Robustness Variance': agg.get('mean_robustness_variance','N/A'),
            'Explanation Align':   'TBD (Phase 3)',
        })

    df = pd.DataFrame(rows).set_index('Model')
    display(df)

In [ ]:
# Per-class robustness breakdown
if EVAL_JSON.exists():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    for ax, (model_name, res) in zip(axes, results.items()):
        per_class = res['aggregate'].get('per_class_robustness', {})
        if not per_class:
            ax.set_title(f'{model_name} — no data yet')
            continue

        classes = list(per_class.keys())
        values  = list(per_class.values())
        colors  = ['#E24B4A' if model_name == 'maskclip' else '#378ADD'] * len(classes)

        ax.barh(classes, values, color=colors, alpha=0.75, edgecolor='none')
        ax.set_xlabel('Robustness Variance (lower = better)', fontsize=10)
        ax.set_title(model_name, fontsize=12, fontweight='bold')
        ax.axvline(0, color='gray', linewidth=0.5)

    plt.suptitle('Per-class Prompt Robustness Variance', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'final' / 'robustness_barchart.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/final/robustness_barchart.png')

---
## 3. Visual Mask Comparisons

Run `scripts/compare_outputs.py` first to generate comparison figures.

In [ ]:
# Display all comparison figures in the notebook
comparison_figs = sorted(COMPARISON_DIR.glob('*.png'))

if not comparison_figs:
    print('[!] No comparison figures found.')
    print('    Run: python scripts/compare_outputs.py --input data/... --prompt "..." --output results/comparison/')
else:
    print(f'Found {len(comparison_figs)} comparison figure(s).\n')
    for fig_path in comparison_figs[:10]:   # show first 10
        img = Image.open(fig_path)
        plt.figure(figsize=(15, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(fig_path.stem, fontsize=10, color='gray')
        plt.tight_layout()
        plt.show()

---
## 4. Prompt Robustness — Side by Side

For each concept, show how mask quality degrades as the prompt becomes more abstract.

In [ ]:
# Plot IoU vs prompt variant for each class and model
if EVAL_JSON.exists():
    import yaml
    with open('../configs/prompt_variants.yaml') as f:
        prompt_cfg = yaml.safe_load(f)

    variant_order = ['base', 'paraphrase', 'abstract', 'vague']
    classes       = list(prompt_cfg['prompts'].keys())

    fig, axes = plt.subplots(1, len(classes), figsize=(4 * len(classes), 4), sharey=True)
    if len(classes) == 1:
        axes = [axes]

    mc_color = '#E24B4A'
    dc_color = '#378ADD'

    for ax, cls in zip(axes, classes):
        for model_name, color in [('maskclip', mc_color), ('denseclip', dc_color)]:
            res = results.get(model_name, {})

            # Aggregate IoU per variant across all images for this class
            variant_ious = {v: [] for v in variant_order}
            for entry in res.get('per_image', []):
                if entry.get('class') == cls:
                    for v in variant_order:
                        val = entry.get('iou_scores', {}).get(v)
                        if val is not None:
                            variant_ious[v].append(val)

            means = [np.mean(variant_ious[v]) if variant_ious[v] else 0
                     for v in variant_order]

            ax.plot(variant_order, means, marker='o', label=model_name,
                    color=color, linewidth=2, markersize=6)

        ax.set_title(cls, fontsize=11, fontweight='bold')
        ax.set_xlabel('Prompt type', fontsize=9)
        ax.set_ylim(0, 1)
        ax.tick_params(axis='x', labelrotation=20, labelsize=8)

    axes[0].set_ylabel('Mean IoU', fontsize=10)
    axes[-1].legend(fontsize=9)
    plt.suptitle('IoU vs Prompt Specificity (base → vague)', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'final' / 'robustness_line_chart.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/final/robustness_line_chart.png')

---
## 5. Failure Case Analysis

Categorize failure modes observed during evaluation.

In [ ]:
# Fill in failure cases manually after running inference
# Categories based on visual inspection of results

failure_cases = [
    {
        'model':    'maskclip',
        'image_id': 'TODO',
        'prompt':   'TODO',
        'category': 'background confusion',   # e.g. mask bleeds onto background
        'note':     'TODO — fill in after running inference',
    },
    {
        'model':    'denseclip',
        'image_id': 'TODO',
        'prompt':   'TODO',
        'category': 'empty mask',              # model predicted nothing
        'note':     'TODO — fill in after running inference',
    },
]

# Failure category taxonomy
categories = [
    'background confusion',   # mask bleeds into background
    'empty mask',             # model predicted nothing
    'wrong object',           # mask on wrong object entirely
    'partial mask',           # correct object but incomplete coverage
    'abstract prompt fail',   # fails only on abstract / vague prompts
]

print('Failure case taxonomy:')
for i, cat in enumerate(categories, 1):
    print(f'  {i}. {cat}')
print('\nFill in failure_cases[] above after running inference.')

---
## 6. Observations & Challenges

> Fill this section in after running experiments. Use the prompts below as a guide.

**MaskCLIP observations:**
- _How well does it handle base prompts vs abstract ones?_
- _Does it over-segment or under-segment?_
- _Any consistent failure patterns?_

**DenseCLIP observations:**
- _Does the ContextDecoder improve spatial grounding over MaskCLIP?_
- _Which prompt types benefit most from the dense head?_
- _Trade-offs vs MaskCLIP in speed and accuracy?_

**Implementation challenges:**
- _Anything unexpected during reimplementation?_
- _Any deviations from the original papers?_

**Next steps (Phase 3):**
- Build LLM explanation module using Grok API
- Crop predicted regions and generate one-sentence justifications
- Compute explanation alignment scores

In [ ]:
# Summary stats for the report
if EVAL_JSON.exists():
    print('=== Midterm Summary ===')
    for model_name, res in results.items():
        agg = res['aggregate']
        print(f'\n{model_name.upper()}')
        print(f'  mIoU:                    {agg.get("miou", "N/A")}')
        print(f'  Mean CLIP similarity:    {agg.get("mean_clip_sim", "N/A")}')
        print(f'  Mean robustness var:     {agg.get("mean_robustness_variance", "N/A")}')
else:
    print('[!] Run evaluation first: python evaluation/evaluate.py --model all')